# EnderLeaf script preparation

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Needed to import from the enderscope library

import os

os.chdir("..")

## Imports

In [ ]:
from rich.pretty import pprint

import numpy as np

from enderleaf.image import to_pil, safe_pil_resize
from enderleaf.draw import image_grid
from enderleaf.enderleaf_ui import ui_show, controller
from enderleaf.enderleaf_ctrl import CameraState
from enderscope.enderlights_pi import CardPoint

import panel as pn

## Initialize Preview

In [ ]:
ui_show()

In [ ]:
controller.launch_acquisition(precise_focusing=False, switch_state=True)

In [ ]:
controls = {
    "AeEnable": False,
    "ExposureTime": 3000,
    "AnalogueGain": 1,
    "AwbEnable": False,
    "ColourGains": (2.3, 0.9),
}

# controls = {
#     "AeEnable": True,
#     "AwbEnable": True,
# }


controller.camera.set_controls(controls)

In [ ]:
controller.set_top_lights(
    True,
    card_points=[
        CardPoint.NORTH,
        CardPoint.SOUTH,
        CardPoint.EAST,
        CardPoint.WEST,
    ],
)

In [ ]:
img_east, _ = controller.capture_array()

In [ ]:
to_pil(img_east)

In [ ]:
to_pil(img_west)

In [ ]:
import numpy as np

image_grid([img_east, img_west,np.minimum(img_east,img_west)], row_count=2)

In [ ]:
image_data = controller.last_job_data

sel_image = pn.widgets.IntSlider(
    name="Select image",
    start=0,
    end=len(image_data) - 1,
    value=0,
    sizing_mode="stretch_width",
)
ph_image = pn.pane.Placeholder()
json_data = pn.pane.JSON()


@pn.depends(sel_image.param.value, watch=True)
def on_index_changed(index):
    image = image_data[index][0]
    metadata = image_data[index][1]
    ph_image.object = safe_pil_resize(to_pil(image), 600, 600)
    json_data.object = {k: str(v) for k, v in metadata.items() if k != "image"}


on_index_changed(sel_image.value)

pn.Column(sel_image, pn.Row(ph_image, json_data))